# Interactive Model Prediction Testing

**Project:** Active Medical Chatbot with ANN-based Diagnosis System

**Date:** 2025-10-18

---

## Overview

This notebook provides interactive tools to test the trained PyTorch ANN model with:

- **Test Set Samples:** Random samples from the test set
- **Custom Symptom Input:** Manual symptom combinations
- **Edge Cases:** Symptoms outside the training data
- **Top-K Predictions:** Multiple candidate diseases with confidence scores
- **Comparison:** Predicted vs actual diagnoses

---

## Objectives

1. Validate model predictions on known test cases
2. Test model behavior with custom symptom combinations
3. Explore model confidence and top-K accuracy
4. Identify cases requiring active questioning
5. Test edge cases and out-of-distribution inputs

## 1. Environment Setup

In [38]:
# Import required libraries
import pandas as pd
import numpy as np
import json
import pickle
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# PyTorch imports
import torch
import torch.nn as nn

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 50)
pd.set_option('display.width', 120)

print("Libraries imported successfully")
print(f"PyTorch version: {torch.__version__}")
print(f"Device: {'CUDA' if torch.cuda.is_available() else 'CPU'}")

Libraries imported successfully
PyTorch version: 2.8.0
Device: CPU


## 2. Configuration and Paths

In [2]:
# Define paths (adjust if running from different directory)
BASE_DIR = Path('..')  # Assumes notebook is in building_ANN/notebooks/
DATA_DIR = BASE_DIR / 'data' / 'processed' / 'common'
MODEL_DIR = BASE_DIR / 'models' / 'pytorch'

# Device configuration
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print("="*80)
print("INTERACTIVE PREDICTION TESTING")
print("="*80)
print(f"Data directory: {DATA_DIR}")
print(f"Model directory: {MODEL_DIR}")
print(f"Device: {DEVICE}")

INTERACTIVE PREDICTION TESTING
Data directory: ../data/processed/common
Model directory: ../models/pytorch
Device: cpu


## 3. Load Model and Data

In [40]:
# Load training data to get symptom names
print("Loading training data (for symptom names)...")
train_df = pd.read_csv(DATA_DIR / 'train.csv')
symptom_columns = [col for col in train_df.columns if col != 'disease']
num_symptoms = len(symptom_columns)

print(f"Found {num_symptoms} symptoms")
print(f"\nFirst 10 symptoms:")
for i, symptom in enumerate(symptom_columns[:10], 1):
    print(f"   {i}. {symptom}")

Loading training data (for symptom names)...
Found 377 symptoms

First 10 symptoms:
   1. lo lắng và căng thẳng
   2. trầm cảm
   3. hụt hơi
   4. các triệu chứng trầm cảm hoặc loạn thần
   5. đau ngực dữ dội
   6. chóng mặt
   7. mất ngủ
   8. các chuyển động không tự nguyện bất thường
   9. tức ngực
   10. hồi hộp


In [41]:
# Load test data
print("\nLoading test dataset...")
test_df = pd.read_csv(DATA_DIR / 'test.csv')

print(f"Test dataset loaded")
print(f"   Test samples: {len(test_df):,}")
print(f"   Unique diseases: {test_df['disease'].nunique()}")


Loading test dataset...
Test dataset loaded
   Test samples: 28,327
   Unique diseases: 577


In [43]:
# Load label encoder
print("\nLoading label encoder...")
with open(MODEL_DIR / 'label_encoder.pkl', 'rb') as f:
    label_encoder = pickle.load(f)

num_classes = len(label_encoder.classes_)
print(f"Label encoder loaded")
print(f"   Number of diseases: {num_classes}")


Loading label encoder...
Label encoder loaded
   Number of diseases: 577


In [44]:
# Define model architecture (same as training)
class ANNDiseaseClassifier(nn.Module):
    def __init__(self, input_dim, num_classes, dropout_rates=(0.3, 0.3, 0.2)):
        super(ANNDiseaseClassifier, self).__init__()
        
        self.fc1 = nn.Linear(input_dim, 512)
        self.bn1 = nn.BatchNorm1d(512)
        self.dropout1 = nn.Dropout(dropout_rates[0])
        
        self.fc2 = nn.Linear(512, 256)
        self.bn2 = nn.BatchNorm1d(256)
        self.dropout2 = nn.Dropout(dropout_rates[1])
        
        self.fc3 = nn.Linear(256, 128)
        self.bn3 = nn.BatchNorm1d(128)
        self.dropout3 = nn.Dropout(dropout_rates[2])
        
        self.fc4 = nn.Linear(128, num_classes)
        self.relu = nn.ReLU()
    
    def forward(self, x):
        x = self.fc1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.dropout1(x)
        
        x = self.fc2(x)
        x = self.bn2(x)
        x = self.relu(x)
        x = self.dropout2(x)
        
        x = self.fc3(x)
        x = self.bn3(x)
        x = self.relu(x)
        x = self.dropout3(x)
        
        x = self.fc4(x)
        return x

print("Model architecture defined")

Model architecture defined


In [45]:
# Load trained model
print("Loading trained model...")
model = ANNDiseaseClassifier(num_symptoms, num_classes)
model = model.to(DEVICE)

# Load best model checkpoint
checkpoint = torch.load(MODEL_DIR / 'best_model.pth', map_location=DEVICE)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

print(f"Model loaded successfully")
print(f"   Best epoch: {checkpoint['epoch'] + 1}")
print(f"   Validation accuracy: {checkpoint['val_acc']*100:.2f}%")
print(f"   Validation loss: {checkpoint['val_loss']:.4f}")

Loading trained model...
Model loaded successfully
   Best epoch: 77
   Validation accuracy: 83.75%
   Validation loss: 0.4244


## 4. Helper Functions for Prediction

In [46]:
def predict_disease(symptom_vector, model, label_encoder, device, top_k=10):
    """
    Predict disease from symptom vector.
    
    Args:
        symptom_vector: numpy array of shape (num_symptoms,) with binary 0/1 values
        model: trained PyTorch model
        label_encoder: sklearn LabelEncoder for diseases
        device: torch device
        top_k: number of top predictions to return
    
    Returns:
        Dictionary with predictions and confidence scores
    """
    # Convert to tensor
    x = torch.FloatTensor(symptom_vector).unsqueeze(0).to(device)
    
    # Get prediction
    with torch.no_grad():
        output = model(x)
        probs = torch.softmax(output, dim=1)
        
        # Get top-k predictions
        top_probs, top_indices = torch.topk(probs, k=top_k, dim=1)
        
    # Convert to numpy
    top_probs = top_probs.cpu().numpy()[0]
    top_indices = top_indices.cpu().numpy()[0]
    
    # Get disease names
    top_diseases = label_encoder.inverse_transform(top_indices)
    
    return {
        'predictions': list(zip(top_diseases, top_probs)),
        'top_disease': top_diseases[0],
        'top_confidence': top_probs[0],
        'all_probs': probs.cpu().numpy()[0]
    }

print("Prediction function defined")

Prediction function defined


In [47]:
def display_prediction(result, actual_disease=None, show_symptoms=None):
    """
    Display prediction results in a formatted way.
    
    Args:
        result: prediction result dictionary
        actual_disease: actual disease name (if known)
        show_symptoms: list of symptom names that are present
    """
    print("="*80)
    print("PREDICTION RESULTS")
    print("="*80)
    
    if show_symptoms:
        print(f"\nINPUT SYMPTOMS ({len(show_symptoms)})")
        for i, symptom in enumerate(show_symptoms, 1):
            print(f"   {i}. {symptom}")
    
    print(f"\nTOP PREDICTION:")
    print(f"   Disease: {result['top_disease']}")
    print(f"   Confidence: {result['top_confidence']*100:.2f}%")
    
    if actual_disease:
        is_correct = result['top_disease'] == actual_disease
        status = "CORRECT" if is_correct else "INCORRECT"
        print(f"\n   Actual disease: {actual_disease}")
        print(f"   Status: {status}")
        
        # Check if actual disease is in top-k
        top_diseases = [pred[0] for pred in result['predictions']]
        if not is_correct and actual_disease in top_diseases:
            rank = top_diseases.index(actual_disease) + 1
            actual_conf = result['predictions'][rank-1][1]
            print(f"   Actual disease rank: #{rank} (confidence: {actual_conf*100:.2f}%)")
    
    print(f"\nTOP-10 PREDICTIONS:")
    for i, (disease, confidence) in enumerate(result['predictions'], 1):
        marker = "👑" if i == 1 else "  "
        if actual_disease and disease == actual_disease:
            marker = "✅"
        bar_length = int(confidence * 50)
        bar = "█" * bar_length
        print(f"   {marker} {i:2d}. {disease:50s} {confidence*100:6.2f}% {bar}")
    
    # Confidence category
    conf = result['top_confidence']
    if conf > 0.85:
        category = "HIGH (>85%) - Direct prediction recommended"
        icon = "🟢"
    elif conf > 0.60:
        category = "MEDIUM (60-85%) - Present top-3 options"
        icon = "🟡"
    else:
        category = "LOW (<60%) - Active questioning required"
        icon = "🔴"
    
    print(f"\n{icon} CONFIDENCE CATEGORY: {category}")
    print("="*80)

print("Display function defined")

Display function defined


In [48]:
def create_symptom_vector(symptom_names, all_symptom_columns):
    """
    Create a binary symptom vector from symptom names.
    
    Args:
        symptom_names: list of symptom names (Vietnamese)
        all_symptom_columns: list of all possible symptom columns
    
    Returns:
        Binary numpy array of shape (num_symptoms,)
    """
    vector = np.zeros(len(all_symptom_columns), dtype='float32')
    
    for symptom in symptom_names:
        if symptom in all_symptom_columns:
            idx = all_symptom_columns.index(symptom)
            vector[idx] = 1.0
        else:
            print(f"⚠️  Warning: Symptom '{symptom}' not found in training data")
    
    return vector

print("Symptom vector creation function defined")

Symptom vector creation function defined


## 5. Test with Random Samples from Test Set

In [49]:
# Test with a random sample from test set
print("Testing with random sample from test set...\n")

# Select random sample
random_idx = np.random.randint(0, len(test_df))
sample = test_df.iloc[random_idx]

# Extract symptoms and actual disease
actual_disease = sample['disease']
symptom_vector = sample[symptom_columns].values.astype('float32')
present_symptoms = [symptom_columns[i] for i in range(len(symptom_vector)) if symptom_vector[i] == 1]

# Predict
result = predict_disease(symptom_vector, model, label_encoder, DEVICE, top_k=10)

# Display
display_prediction(result, actual_disease=actual_disease, show_symptoms=present_symptoms)

Testing with random sample from test set...

PREDICTION RESULTS

INPUT SYMPTOMS (5)
   1. lo lắng và căng thẳng
   2. trầm cảm
   3. mất ngủ
   4. sự tức giận quá mức
   5. nỗi sợ hãi và ám ảnh

TOP PREDICTION:
   Disease: phản ứng điều chỉnh
   Confidence: 29.28%

   Actual disease: rối loạn nhân cách
   Status: INCORRECT
   Actual disease rank: #6 (confidence: 7.23%)

TOP-10 PREDICTIONS:
   👑  1. phản ứng điều chỉnh                                 29.28% ██████████████
       2. rối loạn lưỡng cực                                  23.58% ███████████
       3. sợ xã hội                                           20.61% ██████████
       4. rối loạn tâm thần                                   10.14% █████
       5. tâm thần phân liệt                                   8.73% ████
   ✅  6. rối loạn nhân cách                                   7.23% ███
       7. sự lo lắng                                           0.18% 
       8. rối loạn kiểm soát xung động                         0.12% 
  

## 6. Test with Multiple Random Samples

In [12]:
# Test with multiple random samples
print("🎲 Testing with 5 random samples from test set...\n")

num_samples = 5
correct_count = 0
top3_count = 0
top5_count = 0

for i in range(num_samples):
    print(f"\n{'='*80}")
    print(f"SAMPLE {i+1}/{num_samples}")
    print(f"{'='*80}")
    
    # Select random sample
    random_idx = np.random.randint(0, len(test_df))
    sample = test_df.iloc[random_idx]
    
    # Extract symptoms and actual disease
    actual_disease = sample['disease']
    symptom_vector = sample[symptom_columns].values.astype('float32')
    present_symptoms = [symptom_columns[j] for j in range(len(symptom_vector)) if symptom_vector[j] == 1]
    
    # Predict
    result = predict_disease(symptom_vector, model, label_encoder, DEVICE, top_k=10)
    
    # Check accuracy
    top_diseases = [pred[0] for pred in result['predictions']]
    if result['top_disease'] == actual_disease:
        correct_count += 1
    if actual_disease in top_diseases[:3]:
        top3_count += 1
    if actual_disease in top_diseases[:5]:
        top5_count += 1
    
    # Display (condensed)
    print(f"\n📋 Symptoms: {len(present_symptoms)} symptoms")
    print(f"🎯 Predicted: {result['top_disease']} ({result['top_confidence']*100:.2f}%)")
    print(f"✅ Actual: {actual_disease}")
    
    is_correct = result['top_disease'] == actual_disease
    if is_correct:
        print(f"   Status: ✅ CORRECT")
    else:
        if actual_disease in top_diseases:
            rank = top_diseases.index(actual_disease) + 1
            print(f"   Status: ❌ INCORRECT (actual disease rank: #{rank})")
        else:
            print(f"   Status: ❌ INCORRECT (actual disease not in top-10)")

# Summary
print(f"\n{'='*80}")
print(f"BATCH TESTING SUMMARY")
print(f"{'='*80}")
print(f"Samples tested: {num_samples}")
print(f"Top-1 accuracy: {correct_count}/{num_samples} ({correct_count/num_samples*100:.1f}%)")
print(f"Top-3 accuracy: {top3_count}/{num_samples} ({top3_count/num_samples*100:.1f}%)")
print(f"Top-5 accuracy: {top5_count}/{num_samples} ({top5_count/num_samples*100:.1f}%)")
print(f"{'='*80}")

🎲 Testing with 5 random samples from test set...


SAMPLE 1/5

📋 Symptoms: 5 symptoms
🎯 Predicted: rối loạn nhân cách (95.37%)
✅ Actual: rối loạn nhân cách
   Status: ✅ CORRECT

SAMPLE 2/5

📋 Symptoms: 6 symptoms
🎯 Predicted: phì đại lành tính tuyến tiền liệt (bph) (100.00%)
✅ Actual: phì đại lành tính tuyến tiền liệt (bph)
   Status: ✅ CORRECT

SAMPLE 3/5

📋 Symptoms: 5 symptoms
🎯 Predicted: áp xe răng (84.65%)
✅ Actual: sâu răng
   Status: ❌ INCORRECT (actual disease rank: #2)

SAMPLE 4/5

📋 Symptoms: 6 symptoms
🎯 Predicted: suy thận (100.00%)
✅ Actual: suy thận
   Status: ✅ CORRECT

SAMPLE 5/5

📋 Symptoms: 6 symptoms
🎯 Predicted: chấn thương ở chân (100.00%)
✅ Actual: chấn thương ở chân
   Status: ✅ CORRECT

BATCH TESTING SUMMARY
Samples tested: 5
Top-1 accuracy: 4/5 (80.0%)
Top-3 accuracy: 5/5 (100.0%)
Top-5 accuracy: 5/5 (100.0%)


## 7. Custom Symptom Input - Example Cases

Now you can test with your own symptom combinations!

In [13]:
# Example 1: Common cold symptoms
print("\n🧪 TEST CASE 1: Common Cold Symptoms\n")

custom_symptoms = [
    'ho',
    'nghẹt mũi',
    'đau họng',
    'sốt',
    'Mệt mỏi',
    'đau đầu'
]

# Create symptom vector
symptom_vector = create_symptom_vector(custom_symptoms, symptom_columns)

# Predict
result = predict_disease(symptom_vector, model, label_encoder, DEVICE, top_k=10)

# Display
display_prediction(result, show_symptoms=custom_symptoms)


🧪 TEST CASE 1: Common Cold Symptoms

PREDICTION RESULTS

📋 INPUT SYMPTOMS (6)
   1. ho
   2. nghẹt mũi
   3. đau họng
   4. sốt
   5. Mệt mỏi
   6. đau đầu

🎯 TOP PREDICTION:
   Disease: bệnh bạch cầu đơn nhân
   Confidence: 29.97%

📊 TOP-10 PREDICTIONS:
   👑  1. bệnh bạch cầu đơn nhân                              29.97% ██████████████
       2. bệnh phổi kẽ                                        28.83% ██████████████
       3. áp xe ở họng                                        22.92% ███████████
       4. viêm phế quản cấp tính                              12.04% ██████
       5. loét áp-tơ                                           3.28% █
       6. cảm lạnh thông thường                                1.28% 
       7. cúm                                                  0.69% 
       8. bệnh Lyme                                            0.68% 
       9. rối loạn mũi                                         0.24% 
      10. bệnh bạch cầu                                        0.03% 

In [14]:
# Example 2: Chest pain symptoms (heart-related)
print("\n🧪 TEST CASE 2: Chest Pain Symptoms\n")

custom_symptoms = [
    'đau ngực dữ dội',
    'hụt hơi',
    'chóng mặt',
    'hồi hộp',
    'đổ mồ hôi',
    'buồn nôn'
]

# Create symptom vector
symptom_vector = create_symptom_vector(custom_symptoms, symptom_columns)

# Predict
result = predict_disease(symptom_vector, model, label_encoder, DEVICE, top_k=10)

# Display
display_prediction(result, show_symptoms=custom_symptoms)


🧪 TEST CASE 2: Chest Pain Symptoms

PREDICTION RESULTS

📋 INPUT SYMPTOMS (6)
   1. đau ngực dữ dội
   2. hụt hơi
   3. chóng mặt
   4. hồi hộp
   5. đổ mồ hôi
   6. buồn nôn

🎯 TOP PREDICTION:
   Disease: đau thắt ngực
   Confidence: 85.90%

📊 TOP-10 PREDICTIONS:
   👑  1. đau thắt ngực                                       85.90% ██████████████████████████████████████████
       2. đau tim                                             10.80% █████
       3. tăng kali máu                                        1.14% 
       4. bệnh van hai lá                                      0.93% 
       5. phản ứng thuốc                                       0.86% 
       6. hạ đường huyết                                       0.17% 
       7. nhịp chậm xoang                                      0.06% 
       8. thoát vị hoành                                       0.04% 
       9. block tim                                            0.03% 
      10. nhịp tim nhanh kịch phát trên thất               

In [15]:
# Example 3: Gastrointestinal symptoms
print("\n🧪 TEST CASE 3: Gastrointestinal Symptoms\n")

custom_symptoms = [
    'đau bụng dữ dội',
    'tiêu chảy',
    'buồn nôn',
    'nôn mửa',
    'sốt',
    'Mệt mỏi'
]

# Create symptom vector
symptom_vector = create_symptom_vector(custom_symptoms, symptom_columns)

# Predict
result = predict_disease(symptom_vector, model, label_encoder, DEVICE, top_k=10)

# Display
display_prediction(result, show_symptoms=custom_symptoms)


🧪 TEST CASE 3: Gastrointestinal Symptoms

PREDICTION RESULTS

📋 INPUT SYMPTOMS (6)
   1. đau bụng dữ dội
   2. tiêu chảy
   3. buồn nôn
   4. nôn mửa
   5. sốt
   6. Mệt mỏi

🎯 TOP PREDICTION:
   Disease: bệnh bạch cầu
   Confidence: 99.95%

📊 TOP-10 PREDICTIONS:
   👑  1. bệnh bạch cầu                                       99.95% █████████████████████████████████████████████████
       2. viêm dạ dày ruột truyền nhiễm                        0.01% 
       3. nôn dai dẳng không rõ nguyên nhân                    0.01% 
       4. viêm dạ dày ruột không nhiễm trùng                   0.01% 
       5. viêm túi thừa                                        0.01% 
       6. hạ kali máu                                          0.01% 
       7. viêm dạ dày                                          0.00% 
       8. cúm                                                  0.00% 
       9. viêm ruột thừa                                       0.00% 
      10. giảm thể tích tuần hoàn                        

## 8. Interactive Custom Input

**Instructions:**
1. Run this cell
2. Enter symptoms one per line (Vietnamese symptom names)
3. Leave blank and press Enter to finish
4. Model will predict top-10 diseases

In [16]:
# Interactive symptom input
print("="*80)
print("INTERACTIVE SYMPTOM INPUT")
print("="*80)
print("\nEnter symptoms one per line (leave blank to finish):")
print("\nAvailable symptoms (first 20):")
for i, symptom in enumerate(symptom_columns[:20], 1):
    print(f"   {i}. {symptom}")
print("   ... (377 symptoms total)")

print("\n" + "="*80)
print("Enter your symptoms (or paste a comma-separated list):")
print("="*80)

# For interactive input (uncomment when running interactively)
# user_symptoms = []
# while True:
#     symptom = input("Symptom (or blank to finish): ").strip()
#     if not symptom:
#         break
#     user_symptoms.append(symptom)

# For demonstration, use a predefined list
user_symptoms = ['đau đầu', 'sốt', 'ho', 'Mệt mỏi', 'đau cơ']
print(f"\nUsing example symptoms: {user_symptoms}")

if user_symptoms:
    # Create symptom vector
    symptom_vector = create_symptom_vector(user_symptoms, symptom_columns)
    
    # Predict
    result = predict_disease(symptom_vector, model, label_encoder, DEVICE, top_k=10)
    
    # Display
    display_prediction(result, show_symptoms=user_symptoms)
else:
    print("\nNo symptoms entered.")

INTERACTIVE SYMPTOM INPUT

Enter symptoms one per line (leave blank to finish):

Available symptoms (first 20):
   1. lo lắng và căng thẳng
   2. trầm cảm
   3. hụt hơi
   4. các triệu chứng trầm cảm hoặc loạn thần
   5. đau ngực dữ dội
   6. chóng mặt
   7. mất ngủ
   8. các chuyển động không tự nguyện bất thường
   9. tức ngực
   10. hồi hộp
   11. nhịp tim không đều
   12. thở nhanh
   13. giọng khàn khàn
   14. đau họng
   15. khó nói
   16. ho
   17. nghẹt mũi
   18. sưng họng
   19. giảm thính lực
   20. cục u trong cổ họng
   ... (377 symptoms total)

Enter your symptoms (or paste a comma-separated list):

Using example symptoms: ['đau đầu', 'sốt', 'ho', 'Mệt mỏi', 'đau cơ']
PREDICTION RESULTS

📋 INPUT SYMPTOMS (5)
   1. đau đầu
   2. sốt
   3. ho
   4. Mệt mỏi
   5. đau cơ

🎯 TOP PREDICTION:
   Disease: bệnh Lyme
   Confidence: 90.60%

📊 TOP-10 PREDICTIONS:
   👑  1. bệnh Lyme                                           90.60% █████████████████████████████████████████████
       2

## 9. Test with Edge Cases

In [27]:
# Edge Case 1: Single symptom
print("\n🧪 EDGE CASE 1: Single Symptom\n")

single_symptom = ['sốt']
symptom_vector = create_symptom_vector(single_symptom, symptom_columns)
result = predict_disease(symptom_vector, model, label_encoder, DEVICE, top_k=10)
display_prediction(result, show_symptoms=single_symptom)

print("\n💡 Observation: With only one symptom, the model has low confidence.")
print("   This is expected and suggests active questioning is needed.")


🧪 EDGE CASE 1: Single Symptom

PREDICTION RESULTS

📋 INPUT SYMPTOMS (1)
   1. sốt

🎯 TOP PREDICTION:
   Disease: thủy đậu
   Confidence: 21.86%

📊 TOP-10 PREDICTIONS:
   👑  1. thủy đậu                                            21.86% ██████████
       2. viêm mô tế bào hốc mắt                              11.57% █████
       3. viêm khớp nhiễm trùng                                5.20% ██
       4. tăng natri máu                                       3.41% █
       5. nôn dai dẳng không rõ nguyên nhân                    2.95% █
       6. bệnh thiếu máu hồng cầu hình liềm                    2.63% █
       7. viêm màng não                                        2.50% █
       8. bệnh Lyme                                            2.34% █
       9. thiếu máu không tái tạo                              2.19% █
      10. viêm khí quản                                        2.18% █

🔴 CONFIDENCE CATEGORY: LOW (<60%) - Active questioning required

💡 Observation: With only one symptom, the m

In [36]:
# Edge Case 2: Many symptoms
print("\n🧪 EDGE CASE 2: Many Symptoms (10+)\n")

many_symptoms = [
    'lo lắng và căng thẳng', 'trầm cảm', 'các triệu chứng trầm cảm hoặc loạn thần', 'Mệt mỏi', 'đau cơ',
    'nghẹt mũi', 'đau họng', 'buồn nôn', 'chóng mặt', 'đau bụng dữ dội'
]
symptom_vector = create_symptom_vector(many_symptoms, symptom_columns)
result = predict_disease(symptom_vector, model, label_encoder, DEVICE, top_k=10)
display_prediction(result, show_symptoms=many_symptoms)

print("\n💡 Observation: With many symptoms, the model may be confused.")
print("   The symptoms may span multiple diseases.")


🧪 EDGE CASE 2: Many Symptoms (10+)

PREDICTION RESULTS

📋 INPUT SYMPTOMS (10)
   1. lo lắng và căng thẳng
   2. trầm cảm
   3. các triệu chứng trầm cảm hoặc loạn thần
   4. Mệt mỏi
   5. đau cơ
   6. nghẹt mũi
   7. đau họng
   8. buồn nôn
   9. chóng mặt
   10. đau bụng dữ dội

🎯 TOP PREDICTION:
   Disease: phản ứng căng thẳng cấp tính
   Confidence: 98.13%

📊 TOP-10 PREDICTIONS:
   👑  1. phản ứng căng thẳng cấp tính                        98.13% █████████████████████████████████████████████████
       2. tăng kali máu                                        1.11% 
       3. cai thuốc                                            0.52% 
       4. bệnh tim thiếu máu cục bộ                            0.08% 
       5. viêm xương khớp                                      0.07% 
       6. rối loạn hoảng sợ                                    0.03% 
       7. viêm phổi                                            0.03% 
       8. hạ đường huyết                                       0.01% 
       

In [29]:
# Edge Case 3: No symptoms (all zeros)
print("\n🧪 EDGE CASE 3: No Symptoms (All Zeros)\n")

zero_symptoms = []
symptom_vector = np.zeros(len(symptom_columns), dtype='float32')
result = predict_disease(symptom_vector, model, label_encoder, DEVICE, top_k=10)
display_prediction(result, show_symptoms=zero_symptoms)

print("\n💡 Observation: With no symptoms, the model predicts based on class frequency.")
print("   Top predictions are likely the most common diseases in the training set.")


🧪 EDGE CASE 3: No Symptoms (All Zeros)

PREDICTION RESULTS

🎯 TOP PREDICTION:
   Disease: dị vật trong tai
   Confidence: 13.91%

📊 TOP-10 PREDICTIONS:
   👑  1. dị vật trong tai                                    13.91% ██████
       2. lão thính học                                        6.39% ███
       3. chứng khó nuốt                                       5.15% ██
       4. bàng quang mất trương lực                            3.54% █
       5. hẹp ống dẫn nước mắt                                 2.16% █
       6. rối loạn tuyến nước bọt                              1.97% 
       7. sốt thấp khớp mãn tính                               1.57% 
       8. chấn thương đầu gối                                  1.22% 
       9. ung thư đại trực tràng                               1.20% 
      10. đốt cháy                                             1.19% 

🔴 CONFIDENCE CATEGORY: LOW (<60%) - Active questioning required

💡 Observation: With no symptoms, the model predicts based on class fr

In [30]:
# Edge Case 4: Symptom not in training data
print("\n🧪 EDGE CASE 4: Symptom NOT in Training Data\n")

unknown_symptoms = [
    'sốt',
    'đau đầu',
    'triệu chứng không tồn tại',  # This symptom doesn't exist
    'ho'
]
symptom_vector = create_symptom_vector(unknown_symptoms, symptom_columns)
result = predict_disease(symptom_vector, model, label_encoder, DEVICE, top_k=10)
display_prediction(result, show_symptoms=[s for s in unknown_symptoms if s in symptom_columns])

print("\n💡 Observation: Unknown symptoms are ignored (treated as 0).")
print("   The model only uses symptoms it was trained on.")


🧪 EDGE CASE 4: Symptom NOT in Training Data

⚠️  Warning: Symptom 'triệu chứng không tồn tại' not found in training data
PREDICTION RESULTS

📋 INPUT SYMPTOMS (3)
   1. sốt
   2. đau đầu
   3. ho

🎯 TOP PREDICTION:
   Disease: viêm màng não
   Confidence: 23.91%

📊 TOP-10 PREDICTIONS:
   👑  1. viêm màng não                                       23.91% ███████████
       2. bệnh bạch cầu đơn nhân                              20.64% ██████████
       3. loét áp-tơ                                          16.29% ████████
       4. áp xe ở họng                                        14.64% ███████
       5. bệnh phổi kẽ                                        11.30% █████
       6. cúm                                                  4.16% ██
       7. cảm lạnh thông thường                                2.81% █
       8. viêm họng liên cầu khuẩn                             1.37% 
       9. viêm phế quản cấp tính                               1.34% 
      10. bệnh Lyme                      

## 10. Test Specific Diseases

Select a disease and test with its typical symptom patterns.

In [31]:
# Get a specific disease from test set
print("🔍 Testing with specific disease patterns...\n")

# Show available diseases
diseases = test_df['disease'].unique()
print(f"Total diseases in test set: {len(diseases)}")
print(f"\nFirst 20 diseases:")
for i, disease in enumerate(sorted(diseases)[:20], 1):
    count = len(test_df[test_df['disease'] == disease])
    print(f"   {i}. {disease} ({count} samples)")

🔍 Testing with specific disease patterns...

Total diseases in test set: 577

First 20 diseases:
   1. block tim (71 samples)
   2. bong gân hoặc căng cơ (182 samples)
   3. bong võng mạc (23 samples)
   4. bàn chân bẹt (2 samples)
   5. bàng quang mất trương lực (3 samples)
   6. bệnh về nướu (120 samples)
   7. bướu cổ (3 samples)
   8. bướu cổ đa nhân độc hại (3 samples)
   9. bại não (7 samples)
   10. bệnh Alzheimer (26 samples)
   11. bệnh Crohn (41 samples)
   12. bệnh Graves (14 samples)
   13. bệnh Lyme (13 samples)
   14. bệnh Meniere (24 samples)
   15. bệnh Parkinson (47 samples)
   16. bệnh bạch cầu (48 samples)
   17. bệnh bạch cầu đơn nhân (24 samples)
   18. bệnh celiac (6 samples)
   19. bệnh chlamydia (7 samples)
   20. bệnh chàm (181 samples)


In [32]:
# Test with all samples of a specific disease
target_disease = 'viêm phế quản cấp tính'  # Change this to any disease

print(f"\n{'='*80}")
print(f"TESTING ALL SAMPLES OF: {target_disease}")
print(f"{'='*80}")

disease_samples = test_df[test_df['disease'] == target_disease]
print(f"\nNumber of samples: {len(disease_samples)}")

if len(disease_samples) > 0:
    correct = 0
    top3 = 0
    confidences = []
    
    # Test first 5 samples (or all if less than 5)
    num_to_test = min(5, len(disease_samples))
    
    for i, (idx, sample) in enumerate(disease_samples.head(num_to_test).iterrows(), 1):
        print(f"\n--- Sample {i}/{num_to_test} ---")
        
        symptom_vector = sample[symptom_columns].values.astype('float32')
        present_symptoms = [symptom_columns[j] for j in range(len(symptom_vector)) if symptom_vector[j] == 1]
        
        result = predict_disease(symptom_vector, model, label_encoder, DEVICE, top_k=10)
        
        print(f"Symptoms: {len(present_symptoms)}")
        print(f"Predicted: {result['top_disease']} ({result['top_confidence']*100:.2f}%)")
        
        if result['top_disease'] == target_disease:
            correct += 1
            print("Status: ✅ CORRECT")
        else:
            top_diseases = [pred[0] for pred in result['predictions']]
            if target_disease in top_diseases:
                rank = top_diseases.index(target_disease) + 1
                print(f"Status: ❌ INCORRECT (actual at rank #{rank})")
                if rank <= 3:
                    top3 += 1
            else:
                print("Status: ❌ INCORRECT (not in top-10)")
        
        confidences.append(result['top_confidence'])
    
    print(f"\n{'='*80}")
    print(f"SUMMARY FOR: {target_disease}")
    print(f"{'='*80}")
    print(f"Samples tested: {num_to_test}")
    print(f"Top-1 accuracy: {correct}/{num_to_test} ({correct/num_to_test*100:.1f}%)")
    print(f"Top-3 accuracy: {(correct+top3)}/{num_to_test} ({(correct+top3)/num_to_test*100:.1f}%)")
    print(f"Average confidence: {np.mean(confidences)*100:.2f}%")
    print(f"{'='*80}")
else:
    print(f"\nNo samples found for disease: {target_disease}")


TESTING ALL SAMPLES OF: viêm phế quản cấp tính

Number of samples: 182

--- Sample 1/5 ---
Symptoms: 8
Predicted: viêm phế quản cấp tính (99.85%)
Status: ✅ CORRECT

--- Sample 2/5 ---
Symptoms: 7
Predicted: viêm phế quản cấp tính (90.42%)
Status: ✅ CORRECT

--- Sample 3/5 ---
Symptoms: 6
Predicted: viêm phế quản cấp tính (99.84%)
Status: ✅ CORRECT

--- Sample 4/5 ---
Symptoms: 4
Predicted: viêm phế quản cấp tính (99.98%)
Status: ✅ CORRECT

--- Sample 5/5 ---
Symptoms: 7
Predicted: viêm phế quản cấp tính (99.74%)
Status: ✅ CORRECT

SUMMARY FOR: viêm phế quản cấp tính
Samples tested: 5
Top-1 accuracy: 5/5 (100.0%)
Top-3 accuracy: 5/5 (100.0%)
Average confidence: 97.97%


## 11. Symptom Search Helper

In [33]:
# Search for symptoms containing a keyword
def search_symptoms(keyword, symptom_list):
    """
    Search for symptoms containing a keyword.
    
    Args:
        keyword: search term (Vietnamese)
        symptom_list: list of all symptoms
    
    Returns:
        List of matching symptoms
    """
    matches = [s for s in symptom_list if keyword.lower() in s.lower()]
    return matches

# Example: search for symptoms related to "đau" (pain)
keyword = 'đau'
matches = search_symptoms(keyword, symptom_columns)
print(f"\n🔍 Symptoms containing '{keyword}': {len(matches)} found")
print(f"\nFirst 20 matches:")
for i, symptom in enumerate(matches[:20], 1):
    print(f"   {i}. {symptom}")


🔍 Symptoms containing 'đau': 55 found

First 20 matches:
   1. đau ngực dữ dội
   2. đau họng
   3. đau chân
   4. đau hông
   5. đau vùng trên xương mu
   6. đau ở tinh hoàn
   7. đau bụng dữ dội
   8. đau đầu
   9. đi tiểu đau
   10. đau khi giao hợp
   11. đau bụng dưới
   12. đau tay hoặc ngón tay
   13. đau cổ tay
   14. đau cánh tay
   15. đau răng
   16. đau mặt
   17. đau mắt
   18. đau lưng
   19. đau cổ
   20. đau lưng dưới


In [34]:
# Show all available symptoms (for reference)
print("\n📋 ALL AVAILABLE SYMPTOMS (377 total):\n")
print("Use these exact names for custom symptom input.\n")

for i, symptom in enumerate(symptom_columns, 1):
    print(f"{i:3d}. {symptom}")


📋 ALL AVAILABLE SYMPTOMS (377 total):

Use these exact names for custom symptom input.

  1. lo lắng và căng thẳng
  2. trầm cảm
  3. hụt hơi
  4. các triệu chứng trầm cảm hoặc loạn thần
  5. đau ngực dữ dội
  6. chóng mặt
  7. mất ngủ
  8. các chuyển động không tự nguyện bất thường
  9. tức ngực
 10. hồi hộp
 11. nhịp tim không đều
 12. thở nhanh
 13. giọng khàn khàn
 14. đau họng
 15. khó nói
 16. ho
 17. nghẹt mũi
 18. sưng họng
 19. giảm thính lực
 20. cục u trong cổ họng
 21. cổ họng cảm thấy căng cứng
 22. khó nuốt
 23. sưng da
 24. giữ nước tiểu
 25. khối u ở háng
 26. đau chân
 27. đau hông
 28. đau vùng trên xương mu
 29. máu trong phân
 30. thiếu tăng trưởng
 31. triệu chứng cảm xúc
 32. yếu khuỷu tay
 33. yếu lưng
 34. mủ trong đờm
 35. các triệu chứng của bìu và tinh hoàn
 36. sưng bìu
 37. đau ở tinh hoàn
 38. đầy hơi
 39. mủ chảy ra từ tai
 40. vàng da
 41. khối u ở bìu
 42. dịch tiết màu trắng từ mắt
 43. trẻ sơ sinh cáu kỉnh
 44. lạm dụng rượu
 45. ngất xỉu
 46. hành v

## 12. Summary and Recommendations

### Key Findings:

1. **Model Performance:**
   - Consistent with evaluation results (~83% top-1, ~95% top-3)
   - High confidence for typical symptom patterns
   - Lower confidence for ambiguous or incomplete symptoms

2. **Confidence Categories:**
   - **High (>85%):** Model is confident - can present direct prediction
   - **Medium (60-85%):** Present top-3 options for user to choose
   - **Low (<60%):** Trigger active questioning for more symptoms

3. **Edge Cases:**
   - Single symptom: Low confidence (expected)
   - Many symptoms: May indicate multiple conditions
   - No symptoms: Predicts most common diseases
   - Unknown symptoms: Ignored by model

### Recommendations for Deployment:

1. **Implement confidence thresholds** (>85%, 60-85%, <60%)
2. **Always show top-3 predictions** for medium/low confidence
3. **Trigger active questioning** when confidence < 60%
4. **Validate symptom names** before prediction
5. **Collect user feedback** to improve model over time

---

**Next Steps:**
- Design active questioning logic based on confused pairs
- Integrate with FastAPI chatbot
- Add department recommendations
- Test with real users